# Introduction to Vector Databases with ChromaDB

This notebook introduces the core ideas behind vector databases and walks through a practical example using ChromaDB to store text, generate embeddings, and run semantic similarity search.

## Learning Goals
- Understand what a vector database is and why it is useful
- Learn how embeddings convert text into numeric vectors
- Create a persistent ChromaDB collection
- Add documents with metadata
- Run semantic search queries
- Understand how similarity scores work
- Explore practical use cases like RAG and recommendation systems

## What is a Vector Database?

A vector database stores data as high-dimensional vectors instead of only raw text. Each item is converted into an embedding, which captures its semantic meaning. Similar items end up close to each other in vector space.

This makes vector databases useful for:
- semantic search
- recommendation systems
- retrieval-augmented generation (RAG)
- clustering and similarity analysis

In this notebook, we will create a small ChromaDB collection, add documents, and query for similar text.

---

# STEP 1: Installation

Install the required libraries for ChromaDB and embeddings.

In [ ]:
# Uncomment and run this cell on first use to install libraries
# !pip install -q chromadb sentence-transformers

print("Libraries are ready to import!")

---

# STEP 2: Import Libraries

Import ChromaDB and required modules for working with vector databases.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from pathlib import Path
import os

print(f"ChromaDB version: {chromadb.__version__}")
print("✓ All libraries imported successfully!")

---

# STEP 3: Create Persistent Storage

Set up a local folder to save your vector database so data persists between notebook runs.

In [ ]:
persist_dir = Path("./chroma_db")
persist_dir.mkdir(exist_ok=True)

print(f"✓ Persistence folder: {persist_dir.resolve()}")

---

# STEP 4: Initialize ChromaDB Client

Create a ChromaDB client that connects to your persistent storage.

In [ ]:
client = chromadb.Client(Settings(
    persist_directory=str(persist_dir),
    anonymized_telemetry=False,
))

print("✓ ChromaDB client initialized successfully!")

---

# STEP 5: Set Up Embedding Function

Choose an embedding function to convert text into vectors. ChromaDB's default function works great for getting started.

In [ ]:
embedding_fn = embedding_functions.DefaultEmbeddingFunction()

sample_text = "This is a sample sentence for embedding generation."
sample_embedding = embedding_fn([sample_text])

print("✓ Embedding function loaded!")
print(f"  - Embedding vector length: {len(sample_embedding[0])}")
print(f"  - First 5 values: {sample_embedding[0][:5]}")

---

# STEP 6: Create a Collection

A collection is a container for related documents. It stores document text, IDs, metadata, and embeddings.

The `get_or_create_collection` method ensures the collection is created once and reused on subsequent runs.

In [ ]:
collection = client.get_or_create_collection(
    name="demo_collection",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn,
)

print(f"✓ Collection created: '{collection.name}'")
print(f"  - Current document count: {collection.count()}")

---

# STEP 7: Add Sample Documents

Add documents to the collection. Each document includes:
- text content
- a unique ID
- metadata

In [ ]:
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A machine learning model learns patterns from data.",
    "Vector databases help retrieve semantically similar information.",
    "Cats are independent and curious animals.",
    "A neural network can process text and images.",
    "Deep learning models require large amounts of training data.",
    "Natural language processing powers conversational AI.",
    "Embeddings represent text as vectors in high-dimensional space.",
]

ids = [f"doc_{i}" for i in range(len(documents))]
metadatas = [
    {"topic": "general", "source": "demo"}
    for _ in documents
]

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
)

print(f"✓ Documents added!")
print(f"  - Total documents in collection: {collection.count()}")

---

# STEP 8: Run a Semantic Search Query

This is the core power of vector databases. We search by meaning, not just exact keyword matching.

In [ ]:
query = "Find similar text about machine learning and retrieval systems."

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: {query}")
print("\nTOP MATCHES:\n")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"Document: {doc}")
    print(f"Metadata: {meta}")
    print(f"Distance: {distance:.4f}")
    print(f"Similarity score: {1 - distance:.4f}")
    print("-" * 60)

---

# STEP 9: Understand Similarity Scores

The query returns the documents with the smallest distance to the query vector. A smaller distance usually means stronger semantic similarity.

Typical interpretation:
- distance close to 0 = very similar
- larger distance = less related

This is why vector databases can find similar text even when the exact words are different.

---

# STEP 10: Metadata Filtering

ChromaDB also supports metadata filtering. This is useful when you only want to search a subset of the data.

In [ ]:
collection.update(
    ids=["doc_0"],
    metadatas=[{"topic": "general", "source": "demo", "category": "animals"}],
)

collection.update(
    ids=["doc_1"],
    metadatas=[{"topic": "machine_learning", "source": "demo", "category": "ai"}],
)

collection.update(
    ids=["doc_2"],
    metadatas=[{"topic": "vector_db", "source": "demo", "category": "database"}],
)

query_ai = "Explain machine learning and neural networks"
results_ai = collection.query(
    query_texts=[query_ai],
    n_results=3,
    where={"category": "ai"},
    include=["documents", "metadatas", "distances"],
)

print("AI query:", query_ai)
print("Filtered results (category='ai'):")
for doc, meta, distance in zip(
    results_ai["documents"][0],
    results_ai["metadatas"][0],
    results_ai["distances"][0],
):
    print(f"- {doc} | metadata={meta} | distance={distance:.4f}")

---

# STEP 11: Update and Delete Documents

In real projects, you often update old content or remove outdated records. ChromaDB supports both.

In [ ]:
collection.update(
    ids=["doc_0"],
    documents=["The quick brown fox jumps over the lazy dog — updated for demonstration."],
    metadatas=[{"topic": "general", "source": "demo", "category": "animals", "version": 2}],
)

collection.add(
    documents=["Temporary document for deletion example."],
    ids=["temp_doc"],
    metadatas=[{"topic": "temporary", "source": "demo"}],
)
print("Count before delete:", collection.count())

collection.delete(ids=["temp_doc"])
print("Count after delete:", collection.count())

updated_doc = collection.get(ids=["doc_0"], include=["documents", "metadatas"])
print("Updated document:", updated_doc["documents"][0])

---

# STEP 12: Practical Use Cases

Vector databases are especially valuable in modern AI systems. Some examples:

### 1. Semantic search
Search by meaning instead of exact keywords.

### 2. RAG (Retrieval-Augmented Generation)
An LLM retrieves the most relevant documents from a vector store and uses them as context before answering a user question.

### 3. Recommendations
Find similar products, articles, or users based on embeddings.

### 4. Duplicate detection
Detect near-duplicate records even when they are not word-for-word identical.

### 5. Clustering
Group similar documents into clusters for topic analysis.

---

# STEP 13: Best Practices

When building with vector databases, keep these recommendations in mind:
- clean your text before embedding
- chunk long documents into smaller, meaningful pieces
- use consistent metadata fields across your collection
- choose the right embedding model for your domain
- design queries carefully for better retrieval quality
- keep your embeddings and metadata in sync
- monitor retrieval quality with real examples

---

# STEP 14: Final Takeaway

Vector databases store semantic representations of data and let us search by meaning rather than simple keyword matching. ChromaDB is a lightweight, beginner-friendly option for learning and prototyping.

This small example is only the beginning. In real applications, vector stores are often used to power search systems, AI chatbots, enterprise knowledge bases, and recommendation services.

The key idea is simple: convert text into embeddings, store them in a vector database, and then retrieve semantically relevant results using similarity search.

In [ ]:
print("\n=== Final Summary ===")
print(f"Collection name: {collection.name}")
print(f"Collection count: {collection.count()}")
print("This notebook demonstrated vector storage, embedding generation, semantic search, and metadata filtering.")